# Calendar Dimension Table

Builds the standard calendar table `bharath_dev_adb_p1.silver.calendar` with one row per day from 1940-01-01 to 2030-12-31, including date key, day, week (ISO), month, quarter, half-year, year attributes and common boolean flags.

In [0]:
%sql
CREATE OR REPLACE TABLE bharath_dev_adb_p1.silver.calendar
COMMENT 'Standard calendar dimension: one row per day from 1940-01-01 to 2030-12-31'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS d
),
base AS (
  SELECT
    d,
    weekday(d) + 1 AS iso_dow  -- 1 = Monday ... 7 = Sunday
  FROM dates
)
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT)                         AS date_key,
  d                                                               AS date,
  -- Day
  dayofmonth(d)                                                   AS day_of_month,
  iso_dow                                                         AS day_of_week,          -- ISO: 1=Mon, 7=Sun
  date_format(d, 'EEEE')                                          AS day_name,
  date_format(d, 'EEE')                                           AS day_name_short,
  dayofyear(d)                                                    AS day_of_year,
  CAST(ceil(dayofmonth(d) / 7.0) AS INT)                          AS day_of_week_in_month, -- e.g. 2nd Monday
  -- Week (ISO 8601)
  weekofyear(d)                                                   AS week_of_year,
  year(date_add(d, 4 - iso_dow))                                  AS iso_year,
  CAST(ceil((dayofmonth(d) + weekday(trunc(d, 'MM'))) / 7.0) AS INT) AS week_of_month,
  date_sub(d, iso_dow - 1)                                        AS week_start_date,      -- Monday
  date_add(d, 7 - iso_dow)                                        AS week_end_date,        -- Sunday
  -- Month
  month(d)                                                        AS month,
  date_format(d, 'MMMM')                                          AS month_name,
  date_format(d, 'MMM')                                           AS month_name_short,
  trunc(d, 'MM')                                                  AS month_start_date,
  last_day(d)                                                     AS month_end_date,
  dayofmonth(last_day(d))                                         AS days_in_month,
  date_format(d, 'yyyy-MM')                                       AS year_month,
  year(d) * 100 + month(d)                                        AS year_month_key,
  -- Quarter
  quarter(d)                                                      AS quarter,
  concat('Q', quarter(d))                                         AS quarter_name,
  trunc(d, 'QUARTER')                                             AS quarter_start_date,
  date_sub(add_months(trunc(d, 'QUARTER'), 3), 1)                 AS quarter_end_date,
  concat(year(d), '-Q', quarter(d))                               AS year_quarter,
  year(d) * 10 + quarter(d)                                       AS year_quarter_key,
  datediff(d, trunc(d, 'QUARTER')) + 1                            AS day_of_quarter,
  -- Half year
  CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END                       AS half_year,
  concat(year(d), '-H', CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END) AS year_half,
  -- Year
  year(d)                                                         AS year,
  trunc(d, 'YEAR')                                                AS year_start_date,
  make_date(year(d), 12, 31)                                      AS year_end_date,
  -- Flags
  iso_dow IN (6, 7)                                               AS is_weekend,
  iso_dow NOT IN (6, 7)                                           AS is_weekday,
  (year(d) % 4 = 0 AND year(d) % 100 <> 0) OR year(d) % 400 = 0   AS is_leap_year,
  d = trunc(d, 'MM')                                              AS is_month_start,
  d = last_day(d)                                                 AS is_month_end,
  d = trunc(d, 'QUARTER')                                         AS is_quarter_start,
  d = date_sub(add_months(trunc(d, 'QUARTER'), 3), 1)             AS is_quarter_end,
  d = trunc(d, 'YEAR')                                            AS is_year_start,
  d = make_date(year(d), 12, 31)                                  AS is_year_end
FROM base
ORDER BY d;

In [0]:
%sql
-- Expected: 33,238 rows (91 years x 365 + 23 leap days), unique date_key, range 1940-01-01 to 2030-12-31
SELECT
  count(*)                  AS row_count,
  count(DISTINCT date_key)  AS distinct_date_keys,
  min(date)                 AS min_date,
  max(date)                 AS max_date,
  datediff(max(date), min(date)) + 1 AS expected_days,
  sum(CASE WHEN is_leap_year AND month = 2 AND day_of_month = 29 THEN 1 ELSE 0 END) AS leap_days
FROM bharath_dev_adb_p1.silver.calendar;

In [0]:
%sql
SELECT * FROM bharath_dev_adb_p1.silver.calendar
WHERE date BETWEEN DATE'2026-12-28' AND DATE'2027-01-05'
ORDER BY date;